## Silver pipeline — dedup + MERGE upsert (production)

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("bronze_schema", "lena066636_bronze")
dbutils.widgets.text("bronze_table", "orders_streaming")
dbutils.widgets.text("silver_schema", "lena066636_silver")
dbutils.widgets.text("silver_table", "orders")
dbutils.widgets.text("key_columns", "order_id")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

catalog = dbutils.widgets.get("catalog")
bronze_table = f"{catalog}.{dbutils.widgets.get('bronze_schema')}.{dbutils.widgets.get('bronze_table')}"
silver_table = f"{catalog}.{dbutils.widgets.get('silver_schema')}.{dbutils.widgets.get('silver_table')}"
key_columns = dbutils.widgets.get("key_columns").split(",")


In [0]:
w = Window.partitionBy(*key_columns).orderBy(F.col("ingestion_timestamp").desc())

df_dedup = (spark.table(bronze_table)
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn")
    .withColumn("silver_load_timestamp", F.current_timestamp()))


In [0]:
merge_condition = " AND ".join([f"t.{c} = s.{c}" for c in key_columns])

if spark.catalog.tableExists(silver_table):
    target = DeltaTable.forName(spark, silver_table)
    (target.alias("t")
        .merge(df_dedup.alias("s"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
else:
    df_dedup.write.format("delta").saveAsTable(silver_table)
